## Modeling

In [1]:
DATA_FILE = "Data/all_dialogs.txt"

all_dialogs = open(DATA_FILE)

text = all_dialogs.read()

dialogs = text.split("-" * 100)

print(len(dialogs))

3236


In [2]:
#print(dialogs[0])

## Implementation

In [3]:
def create_dialog_state():
    dialog_state =  {
        "state": 1,
        "area": None,
        "food": None,
        "pricerange": None,
        "restaurant": None,
        "alternatives": [],
        "last_response": None,
        "confirmation" : None
    }
    
    return dialog_state


create_dialog_state()

{'state': 1,
 'area': None,
 'food': None,
 'pricerange': None,
 'restaurant': None,
 'alternatives': [],
 'last_response': None,
 'confirmation': None}

### Slot extraction

In [4]:
import pandas as pd

RESTAURANT_FILE = "Data/restaurant_info.csv"

restaurants = pd.read_csv(RESTAURANT_FILE)

#print(restaurants.columns)
#restaurants.head()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\Lucas\anaconda3\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Lucas\anaconda3\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\Lucas\anaconda3\lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\Lucas\anaconda3\lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\Lucas\anaconda3\lib\site-pack

AttributeError: _ARRAY_API not found

In [5]:
areas = restaurants["area"].dropna().unique().tolist()
foods = restaurants["food"].dropna().unique().tolist()
price_ranges = restaurants["pricerange"].dropna().unique().tolist()

#print(areas)
#print(price_ranges)
#print(foods)

In [6]:
### 1. Keyword matching

def keyword_slot_extractor(utterance, current_state):
    utterance = utterance.lower()
    
    slots = {
        "area": None,
        "food": None,
        "pricerange": None
    }
    
    for area in areas:
        if area in utterance:
            slots["area"] = area

    for food in foods:
        if food in utterance:
            slots["food"] = food

    for price in price_ranges:
        if price in utterance:
            slots["pricerange"] = price
            
            
            
    ### Dontcare part (If the user doesnt have a preference), this maybe has to be made more clean/efficient or different
    ### I do not think it is supposed to be this hard-coded?
    
    if "any area" in utterance or "any part of town" in utterance or "any location" in utterance:
        slots["area"] = "dontcare"

    if "any price" in utterance:
        slots["pricerange"] = "dontcare"

    if "any food" in utterance:
        slots["food"] = "dontcare"
        

    if (utterance.strip() == "any" 
        or "i don't care" in utterance
        or "i dont care" in utterance 
        or "dont care" in utterance
        or "it doesn't matter" in utterance
        or "it doesnt matter" in utterance):

        if current_state == 3:
            slots["food"] = "dontcare"

        elif current_state == 4:
            slots["area"] = "dontcare"

        elif current_state == 5:
            slots["pricerange"] = "dontcare"
            
            
            
    
    return slots

In [7]:
### 2. Levenshtein edit distance

#%pip install levenshtein
from Levenshtein import distance

def levenshtein_slot_extractor(utterance, threshold=2):  # Maybe the threshold is better at 1
    utterance = utterance.lower()
    words = utterance.split()

    slots = {
        "area": None,
        "food": None,
        "pricerange": None
    }
    
    # Area
    best_match = None
    best_distance = float("inf")

    for area in areas:
        for word in words:
            current_distance = distance(word, area)

            if current_distance < best_distance:
                best_distance = current_distance
                best_match = area

    if best_distance <= threshold:
        slots["area"] = best_match        
    
    
    # Food
    best_match = None
    best_distance = float("inf")

    for food in foods:
        for word in words:
            current_distance = distance(word, food)

            if current_distance < best_distance:
                best_distance = current_distance
                best_match = food

    if best_distance <= threshold:
        slots["food"] = best_match
    

    # Pricerange
    best_match = None
    best_distance = float("inf")

    for price in price_ranges:
        for word in words:
            current_distance = distance(word, price)

            if current_distance < best_distance:
                best_distance = current_distance
                best_match = price

    if best_distance <= threshold:
        slots["pricerange"] = best_match

    return slots

In [8]:
### 3. Semantic similarity via embeddings

## TODO: Alessandro

### Response generation

Use string templates for all system utterances, either simple strings or patterns with variables to fill in. For example:

"askpricerange": "Which price range do you prefer?"
"confirmfoodtype": "I did not recognize {givenfoodtype}. Did you mean {correctedfoodtype}?"

In [9]:
## TODO: Niousha 

# I made a start already, otherwise I could not go on with the state_transition. (Lucas)

responses = {
    "askpreferences": "What kind of restaurant are you looking for?",
    "askfoodtype": "What kind of food would you like?",
    "askarea": "In which area would you like to eat?",
    "askpricerange": "What is your prefered pricerange?",
    "nomatch": "Sorry, there is no restaurant that matches your current preferences.",
    "goodbye": "Goodbye!",
    "null" : "Sorry, I did not understand that. Could you repeat it?",
    "unknownpreference": "Sorry, I am not familiar with this preference. Would you like something else?"
}


def response_generation(system_utterance, restaurant=None, requested=None, food=None, area=None, pricerange=None, suggested_value=None):
    
    if system_utterance == "recommendrestaurant":
        return f"Based on your preferences I would recommend {restaurant['restaurantname']}."

    
    if system_utterance == "provide":

        restaurant_name = restaurant["restaurantname"]
        value = restaurant[requested]

        if pd.isna(value):
            return f"The {requested} of {restaurant_name} is unknown."

        if requested == "phone":
            return f"The phone number of {restaurant_name} is {value}."

        elif requested == "addr":
            return f"{restaurant_name} is located at {value}."

        elif requested == "postcode":
            return f"The postcode of {restaurant_name} is {value}."
        
    
    if system_utterance == "askconfirmation":
        return f"Did you mean {suggested_value}?"
    
    
    
    if system_utterance == "confirmproperty":
        actual_value = restaurant[requested]

        if suggested_value == actual_value:
            return "Yes, that's correct."
        else:
            return f"No, the {requested} is actually {actual_value}."

    

    if system_utterance in responses:
        return responses[system_utterance]

    return "ERROR in response_generation"
    

### Restaurant lookup

Once all preferences are collected, the system provides a recommendation. Write a Python function that retrieves matching restaurants from restaurant_info.csv according to the extracted preferences. If no restaurant matches, the system should inform the user. If multiple match, one is chosen at random and the others are stored so the user can ask for an alternative using the same preferences.

In [10]:
import random
import pandas as pd

restaurants = pd.read_csv("Data/restaurant_info.csv")

def restaurant_lookup(area, food, pricerange):

    matches = restaurants

    if area != "dontcare":
        matches = matches[matches["area"] == area]

    if food != "dontcare":
        matches = matches[matches["food"] == food]

    if pricerange != "dontcare":
        matches = matches[matches["pricerange"] == pricerange]
        
        
    matches = matches.to_dict("records")

    if len(matches) == 0:
        return None, []
    
    recommendation = random.choice(matches)
    
    alternatives = [restaurant for restaurant in matches if restaurant != recommendation]
        
    return recommendation, alternatives


#recommendation, alternatives = restaurant_lookup("north", "indian", "moderate")
#print(recommendation)
#print(alternatives)

#print(restaurants)
#print(recommendation["restaurantname"])

In [11]:
# Not sure if they want this to be in the same function as keyword_slot_extractor but for now I will do it seperate

def request_extractor(utterance):
    utterance = utterance.lower()

    if "phone" in utterance:
        return "phone"

    elif "address" in utterance:
        return "addr"

    elif "postcode" in utterance or "post code" in utterance or "postal code" in utterance:
        return "postcode"

    elif "food" in utterance:
        return "food"

    elif "area" in utterance:
        return "area"

    elif "price" in utterance:
        return "pricerange"

### Reasoning component

In [12]:
### TODO: Add this to the main state transition / implementation

# EVA
RESTAURANT_FILE2 = "Data/restaurant_info_extended.csv"

restaurants_extended = pd.read_csv(RESTAURANT_FILE2)

consequents=["touristic", "assigned seats", "children", "romantic"]

def antecedent_rules(consequent):
    if consequent == "touristic":
        true={"pricerange": "cheap",
             "food quality": "good"}
        false= {"food": "romanian"}
    elif consequent == "assigned seats":
        true={"crowdedness": "busy"}
        false= {}
    elif consequent == "children":
        true={}
        false={"length of stay": "long"}
    elif consequent == "romantic":
        true={"crowdedness": "busy"}
        false= {"length of stay": "long"}
    return true, false

def reasoning_component(restaurants_options, consequent):
    true, false = antecedent_rules(consequent)
    matches=[]
    half_match=[]

    for restaurant_option in restaurants_options:
            restaurant = restaurants_extended[restaurants_extended["restaurantname"] == restaurant_option]
            restaurant = restaurant.iloc[0]
        
            positive_match = all(restaurant[attribute] == value
                                 for attribute, value in true.items())
        
            negative_match = all(restaurant[attribute] != value
                                 for attribute, value in false.items())

            if positive_match and negative_match:
                matches.append(restaurant)

            recommendation = random.choice(matches)
        
    return recommendation["restaurantname"]

            # contradiction:
            # if positive_match and not(negative_match):
            #     half_match.append(restaurant)
            
    #return half_match

# recommendation=['frankie and bennys']
# candidates=['the nirala', 'tandoori palace', 'graffiti']
# con="romantic"

# matchings = reasoning_component(candidates, con)

# print(matchings)


## Implementation / Configurability

In [13]:
# state transition function

def state_transition(dialog_state, utterance, dialog_act, fallback="levenshtein"):
    
    current_state = dialog_state["state"]
    next_dialog_state = current_state
    system_utterance = None
    
    
    # Confirmation part
    if current_state == 6 and dialog_state["confirmation"] is not None:
        slot = dialog_state["confirmation"]["slot"]
        value = dialog_state["confirmation"]["value"]

        if dialog_act == "affirm":
            dialog_state[slot] = value
            dialog_state["confirmation"] = None

        elif dialog_act in ["negate", "deny"]:
            dialog_state["confirmation"] = None
                
        else:
            system_utterance = dialog_state["last_response"]
    
    
    
    if dialog_act == "restart":
        dialog_state = create_dialog_state()
        next_dialog_state = 2
        system_utterance = response_generation("askpreferences")

    elif dialog_act in ["bye", "thankyou"]:        
        next_dialog_state = 10  
        system_utterance = response_generation("goodbye")

    elif dialog_act == "repeat":
        system_utterance = dialog_state["last_response"]

    elif dialog_act ==  "null":
        system_utterance = response_generation("null")
    
    elif current_state == 1 and dialog_act == "hello":
        next_dialog_state = 2
        system_utterance = response_generation("askpreferences")
        
    
    
    elif current_state in [1, 2, 3, 4, 5, 7] and dialog_act in ["inform", "reqalts"]:  

        # Keyword slot extractor
        slots = keyword_slot_extractor(utterance, current_state)

        if slots["food"] is not None:
            dialog_state["food"] = slots["food"]

        if slots["area"] is not None:
            dialog_state["area"] = slots["area"]

        if slots["pricerange"] is not None:
            dialog_state["pricerange"] = slots["pricerange"]
            
            
        # Fallback levenshtein extractor
        if fallback == "levenshtein":
            fallback_slots = levenshtein_slot_extractor(utterance)

        ### TODO: ADD THIS
        
        #elif fallback == "semantic":
            #fallback_slots = semantic_slot_extractor(utterance)
            
            
        # Confirmation part
        for slot in ["food", "area", "pricerange"]:
            if (slots[slot] is None and dialog_state[slot] is None and fallback_slots[slot] is not None):
                
                dialog_state["confirmation"] = {"slot": slot, "value": fallback_slots[slot]}

                next_dialog_state = 6
                system_utterance = response_generation("askconfirmation", suggested_value=fallback_slots[slot])

                dialog_state["state"] = next_dialog_state
                dialog_state["last_response"] = system_utterance

                return dialog_state, system_utterance
            
        
        # If an "inform" occurs but the preference does not gets gound. Reference_dialog 6: "I would like some Swedish food."
        if (all(value is None for value in slots.values()) and all(value is None for value in fallback_slots.values())):
            system_utterance = response_generation("unknownpreference")
            
            
            
            
    elif current_state in [8, 9] and dialog_act in ["reqalts", "reqmore"]:
        
        if len(dialog_state["alternatives"]) > 0:
            recommendation = dialog_state["alternatives"].pop(0)

            dialog_state["restaurant"] = recommendation

            next_dialog_state = 8
            system_utterance = response_generation("recommendrestaurant", restaurant=recommendation)

        else:
            next_dialog_state = 7
            system_utterance = response_generation("nomatch")      
            
            
    elif current_state in [8, 9] and dialog_act == "request":

        requested = request_extractor(utterance)

        if requested is not None:
            next_dialog_state = 9

            system_utterance = response_generation("provide", restaurant=dialog_state["restaurant"], requested=requested)         
            
            
            
    elif current_state in [8, 9] and dialog_act == "confirm":
        
        slots = keyword_slot_extractor(utterance, current_state)

        for slot in ["food", "area", "pricerange"]:

            if slots[slot] is not None:
                system_utterance = response_generation("confirmproperty", restaurant=dialog_state["restaurant"], requested=slot, suggested_value=slots[slot])

            
    
            
    if system_utterance is None and dialog_state["confirmation"] is None:

        if (dialog_state["food"] is not None and dialog_state["area"] is not None and dialog_state["pricerange"] is not None):

            recommendation, alternatives = restaurant_lookup(dialog_state["area"], dialog_state["food"], dialog_state["pricerange"])

            if recommendation is not None:
                dialog_state["restaurant"] = recommendation
                dialog_state["alternatives"] = alternatives

                next_dialog_state = 8                               
                system_utterance = response_generation("recommendrestaurant", restaurant=recommendation)

            else:
                next_dialog_state = 7                            
                system_utterance = response_generation("nomatch")



        elif dialog_state["food"] is None:
            next_dialog_state = 3                                    
            system_utterance = response_generation("askfoodtype")

        elif dialog_state["area"] is None:
            next_dialog_state = 4                                    
            system_utterance = response_generation("askarea")

        elif dialog_state["pricerange"] is None:
            next_dialog_state = 5                                    
            system_utterance = response_generation("askpricerange")
            
            
                
                
    if system_utterance is None:
        system_utterance = response_generation("null")
                
                
    dialog_state["state"] = next_dialog_state
    dialog_state["last_response"] = system_utterance
    
    return(dialog_state, system_utterance)

# Run the dialog

In [14]:
############################################################################################################################
####### TODO: WE HAVE TO USE THE BEST CLASSIFIER FROM 1A, NOT THIS ONE.  ##################################################
############################################################################################################################
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from preprocess import load_and_preprocess_data

DATA_PATH = "Data/dailog_acts.dat"

dialog_acts, utterances = load_and_preprocess_data(DATA_PATH)

dialog_vectorizer = CountVectorizer()
X_dialog = dialog_vectorizer.fit_transform(utterances)

dialog_classifier = LinearSVC(
    max_iter=5000,
    random_state=42
)

dialog_classifier.fit(X_dialog, dialog_acts)


def classify_utterance(utterance):

    utterance = utterance.strip().lower()

    if utterance == "":
        return "null"

    vector = dialog_vectorizer.transform([utterance])

    return dialog_classifier.predict(vector)[0]

################################################################################################################################



def run_dialog(debug=False):
    dialog_state = create_dialog_state()

    print("System: Hello, welcome to the Cambridge restaurant recommendation system. How may I help you?")

    while dialog_state["state"] != 10:

        utterance = input("User: ").strip().lower()

        dialog_act = classify_utterance(utterance)

        if debug:
            print(f"[Predicted dialog act: {dialog_act}]")

        dialog_state, response = state_transition(dialog_state, utterance, dialog_act)

        print("System:", response)

    return dialog_state    
    

In [15]:
run_dialog(debug=True)

System: Hello, welcome to the Cambridge restaurant recommendation system. How may I help you?
User: bye
[Predicted dialog act: bye]
System: Goodbye!


{'state': 10,
 'area': None,
 'food': None,
 'pricerange': None,
 'restaurant': None,
 'alternatives': [],
 'last_response': 'Goodbye!',
 'confirmation': None}